In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 16.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which

In [ ]:
!pip install pytorch-lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.0/823.0 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 961.5/961.5 kB 48.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import itertools
from sklearn.model_selection import train_test_split
import torch.optim as optim
from torchsummary import summary
import torch.optim as optim
import pytorch_lightning as pl
from PIL import Image
import wandb
from torchmetrics import Accuracy, F1Score, Precision, Recall
from torchvision import datasets, transforms
from datasets import load_dataset
from torch.utils.data import DataLoader, random_split, Dataset
import torchvision.transforms as T

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
trainset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

100%|██████████| 170M/170M [00:06<00:00, 28.1MB/s]


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pytorch_lightning as pl
from torchmetrics.classification import Accuracy, F1Score, Precision, Recall

class CNN(pl.LightningModule):
    def __init__(self,
                 kernel_size=(3, 3),
                 conv_filters=None,
                 num_classes=10,
                 lr=1e-3,
                 optimizer_cls=None,
                 optimizer_params=None,
                 batch_size=64,
                 dropout=0.5,
                 scheduler_params=None,
                 input_channels=3,
                 dense_layers = [240,60]):
        super().__init__()
        if conv_filters is None:
            conv_filters = [32, 64, 128]
        if optimizer_params is None:
            optimizer_params = {'weight_decay': 1e-4}
        if scheduler_params is None:
            scheduler_params = {'mode': 'min', 'factor': 0.1, 'patience': 3}
        if optimizer_cls is None:
            optimizer_cls = optim.AdamW
        self.dense_layers = dense_layers

        self.save_hyperparameters(ignore=['optimizer_params', 'scheduler_params'])
        self.optimizer_params = optimizer_params
        self.scheduler_params = scheduler_params


        layers = []
        in_channels = input_channels
        for out_channels in conv_filters:
            layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, padding=1))
            layers.append(nn.BatchNorm2d(out_channels))
            layers.append(nn.ReLU())
            layers.append(nn.MaxPool2d(2))
            layers.append(nn.Dropout2d(dropout))
            in_channels = out_channels
        self.feature_extractor = nn.Sequential(*layers)


        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )
        cur_size = conv_filters[-1]
        for i in range(len(self.dense_layers)):
          self.classifier.append(nn.Linear(cur_size,dense_layers[i]))
          cur_size = self.dense_layers[i]
        self.classifier.append(nn.Linear(self.dense_layers[-1], num_classes))
        self.criterion = nn.CrossEntropyLoss()


        self.train_accuracy = Accuracy(task="multiclass", num_classes=num_classes)
        self.train_f1 = F1Score(task="multiclass", num_classes=num_classes)
        self.train_precision = Precision(task="multiclass", num_classes=num_classes)
        self.train_recall = Recall(task="multiclass", num_classes=num_classes)

        self.val_accuracy = Accuracy(task="multiclass", num_classes=num_classes)
        self.val_f1 = F1Score(task="multiclass", num_classes=num_classes)
        self.val_precision = Precision(task="multiclass", num_classes=num_classes)
        self.val_recall = Recall(task="multiclass", num_classes=num_classes)

    def forward(self, x):
        x = self.feature_extractor(x)
        x = self.classifier(x)
        return x

    def compute_metrics(self, logits, targets, prefix=""):
        """
        Вычисляет метрики по предсказаниям и целевым значениям.
        Логиты сначала преобразуются в предсказания с помощью argmax.

        Args:
            logits (Tensor): Выход модели.
            targets (Tensor): Целевые метки.
            prefix (str): Префикс, определяющий, для какого этапа считаем метрики ('train' или 'val').

        Returns:
            dict: Словарь с loss и вычисленными метриками.
        """
        loss = self.criterion(logits, targets)
        preds = torch.argmax(logits, dim=1)
        if prefix == "train":
            acc = self.train_accuracy(preds, targets)
            f1 = self.train_f1(preds, targets)
            prec = self.train_precision(preds, targets)
            rec = self.train_recall(preds, targets)
        else:  # validation
            acc = self.val_accuracy(preds, targets)
            f1 = self.val_f1(preds, targets)
            prec = self.val_precision(preds, targets)
            rec = self.val_recall(preds, targets)
        return {
            "loss": loss,
            "acc": acc,
            "f1": f1,
            "precision": prec,
            "recall": rec
        }

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        metrics = self.compute_metrics(logits, y, prefix="train")
        self.log_dict({f"train_{k}": v for k, v in metrics.items()},
                      on_step=False, on_epoch=True, prog_bar=True)
        return metrics["loss"]

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        metrics = self.compute_metrics(logits, y, prefix="val")
        self.log_dict({f"val_{k}": v for k, v in metrics.items()},
                      on_step=False, on_epoch=True, prog_bar=True)
        return metrics["loss"]

    def cosine_annealing_scheduler(self, optimizer):
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

    def configure_optimizers(self):
        optimizer = self.hparams.optimizer_cls(self.parameters(), lr=self.hparams.lr, **self.optimizer_params)
        scheduler = self.cosine_annealing_scheduler(optimizer)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch",
                "frequency": 1
            }
        }


In [ ]:
model = CNN(
    kernel_size=(3, 3),
    conv_filters=[32, 64, 128],
    num_classes=10,
    input_channels=3,
    lr=1e-3,
    dropout=0.3
)
wandb_logger = pl.loggers.WandbLogger(
        )

trainer = pl.Trainer(
        max_epochs=15,
        logger=wandb_logger,
        accelerator="auto",
        devices=1,
        callbacks=[
            pl.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=5,
                mode="min"
            )
        ],
        gradient_clip_val=1.0,
        enable_progress_bar=True,
        check_val_every_n_epoch=1,
    )
trainer.fit(model, trainloader, testloader)

INFO:pytorch_lightning.utilities.rank_zero:You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
   | Name              | Type                | Params | Mode 
-------------------------------------------------------------------
0  | feature_extractor | Sequential          | 93.7 K | train
1  | classifier        | Sequential          | 46.0 K | train
2  | criterion         | CrossEntropyLoss    | 0      | train
3  | train_accuracy    | MulticlassAccuracy  | 0      | train
4  | train_f1          | MulticlassF1Score   | 0      | train
5  | train_precision   | MulticlassPrecision | 0      | train
6  | train_recall      | MulticlassRecall    | 0      | train
7  | val_accuracy      | MulticlassAccuracy  | 0      | train
8  | val_f1            | MulticlassF1Score   | 0      | train
9  | val_precision     | MulticlassPrecision | 0      | train
10 | val_recall        | MulticlassRecall    | 0      | train
---------------------------------------------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=15` reached.


In [ ]:
dataset = load_dataset('Marxulia/asl_sign_languages_alphabets_v03')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/906 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/74.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10873 [00:00<?, ? examples/s]

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 10873
    })
})

In [ ]:
IMG_SIZE = (32, 32)

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_transforms = T.Compose([
    T.Resize(IMG_SIZE),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize(MEAN, STD)
])

val_test_transforms = T.Compose([
    T.Resize(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(MEAN, STD)])

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        """
        Args:
            hf_dataset: Загруженный датасет Hugging Face (один сплит, например, train).
            transform (callable, optional): Трансформации, которые будут применены к изображению.
        """
        self.hf_dataset = hf_dataset
        self.transform = transform
        self.image_column = 'image'
        self.label_column = 'label'

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        item = self.hf_dataset[idx]
        image = item[self.image_column]
        if self.transform:
            image = self.transform(image)
        label = item[self.label_column]
        label = torch.tensor(label, dtype=torch.long)
        return image, label

In [ ]:
dataset = ImageDataset(dataset['train'], transform=train_transforms)

In [ ]:
train_dataset,test_dataset = random_split(dataset,[0.8,0.2])
train_loader, test_loader = DataLoader(train_dataset),DataLoader(test_dataset)

In [ ]:
filters = [[32,64,128],[64,128,256]]
kernel_sizes = [(5,5)]
batch_sizes = [64, 128]
flatten_sizes = [[512,256,128,64],[256,128,64]]
configs = []
for kernel_size in kernel_sizes:
    for filter in filters:
        for batch_size in batch_sizes:
            for flatten_size in flatten_sizes:
                configs.append((filter, kernel_size, batch_size, flatten_size))

In [ ]:
configs

[([32, 64, 128], (5, 5), 64, [512, 256, 128, 64]),
 ([32, 64, 128], (5, 5), 64, [256, 128, 64]),
 ([32, 64, 128], (5, 5), 128, [512, 256, 128, 64]),
 ([32, 64, 128], (5, 5), 128, [256, 128, 64]),
 ([64, 128, 256], (5, 5), 64, [512, 256, 128, 64]),
 ([64, 128, 256], (5, 5), 64, [256, 128, 64]),
 ([64, 128, 256], (5, 5), 128, [512, 256, 128, 64]),
 ([64, 128, 256], (5, 5), 128, [256, 128, 64])]

In [ ]:
def train_experiment(kernel_size, filter, batch_size, flatten_size):
    wandb.init(
        project="ASL_(5,5)",
        group=f"kernel_size_{kernel_size}",
        name=f"filter_{filter[0]}_bs_{batch_size}_kernel_size_{kernel_size}_flatten_size_{flatten_size[0]}",
        config={
            "batch_size": batch_size,
            "kernel_size": kernel_size,
            "filter": filter
        }
    )

    model = CNN(
      kernel_size=kernel_size,
      conv_filters=filter,
      num_classes=29,
      input_channels=3,
      lr=1e-4,
      dropout=0.3,
      dense_layers = flatten_size
  )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True
    )

    val_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True
    )

    wandb_logger = pl.loggers.WandbLogger(
        log_every_n_steps=0
        )

    trainer = pl.Trainer(
        max_epochs=30,
        logger=wandb_logger,
        accelerator="auto",
        devices=1,
        callbacks=[
            pl.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=5,
                mode="min"
            )
        ],
        gradient_clip_val=1.0,
        enable_progress_bar=True,
        check_val_every_n_epoch=1,
    )

    try:
        trainer.fit(model, train_loader, val_loader)
    except Exception as e:
        print(f"Ошибка в эксперименте kernel_size={kernel_size}, filter={filter[0]}: {str(e)}")
    finally:
        wandb.finish()

In [ ]:
for filter,kernel_size,batch_size, flatten_size in configs:
  train_experiment(kernel_size = kernel_size,filter=filter,batch_size = batch_size, flatten_size = flatten_size)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: bigrussianbossp709 (code_summarization_trial1) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
INFO:pytorch_lightning.utilities.rank_zero:You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/dist-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=30` reached.


epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
train_acc,▁▁▂▃▃▃▄▄▅▅▅▅▆▆▆▇▇▇▇▇▇█████████
train_f1,▁▁▂▃▃▃▄▄▅▅▅▅▆▆▆▇▇▇▇▇▇█████████
train_loss,█▇▇▇▆▆▅▅▅▄▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
train_precision,▁▁▂▃▃▃▄▄▅▅▅▅▆▆▆▇▇▇▇▇▇█████████
train_recall,▁▁▂▃▃▃▄▄▅▅▅▅▆▆▆▇▇▇▇▇▇█████████
trainer/global_step,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
val_acc,▁▂▂▃▄▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████
val_f1,▁▂▂▃▄▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████
val_loss,██▇▇▆▆▅▅▄▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_precision,▁▂▂▃▄▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████
